# CivicLoop Step 3.5: triage LLM (LoRA SFT, then DPO)

Same two stages as LightGBM v1 -> v2, on Qwen2.5-1.5B-Instruct:
1. **SFT** on the rubric-labelled training rows (what v1 learned from).
2. **DPO** on the officials' severity overrides (chosen = official, rejected = model) plus as many anchor pairs (accepted cases far from a school: chosen = accepted severity, rejected = the least-rejected other severity), with the SFT model as the frozen reference. Run 3 adds an SFT loss on the chosen answer, lr 1e-5 and 1 epoch, and stops early if one severity takes over 60% of 150 eval answers (run 1 collapsed to "high", run 2 to "critical").
3. Greedy answers for the triage test split and the override holdout. Scoring happens locally in `python -m trainer.llm_eval`.

**How to run** (about 30-45 min on a free T4):
1. Locally: `.venv/bin/python -m trainer.llm_data`. This writes 3 files to `data/trainer/llm/`.
2. Colab: Runtime -> Change runtime type -> **T4 GPU**. Run all cells and upload the 3 files when asked.
   Kaggle: Settings -> **Internet on**, Settings -> Accelerator -> a GPU, Add Input -> a dataset holding the 3 files.
   The config cell finds the files by itself. Use Run -> Restart & clear cell outputs (never Factory reset) if you restart.
3. Download `triage_llm.zip` (Colab downloads it itself; on Kaggle open the Output panel -> /kaggle/working -> triage_llm.zip), unzip it into `models/trainer/triage_llm/`, then run `.venv/bin/python -m trainer.llm_eval`.

Plain LoRA on an fp16 base, not 4-bit QLoRA. 1.5B in fp16 is about 3 GB, so 4-bit saves nothing that matters on a 15 GB T4. Also, TRL's QLoRA path casts the adapters to bf16, which a T4 lacks.

In [ ]:
# Kaggle needs Settings -> Internet ON for this cell. The bigframes warning afterwards is harmless.
# torchao is not used here, and Kaggle's old copy (0.10.0) makes peft raise ImportError, so remove it.
%pip uninstall -y -q torchao
%pip install -q "transformers==5.19.0" "trl==1.15.0" "peft==0.21.2" "datasets==5.1.0" "accelerate==1.15.0"

In [ ]:
import json
import os
import random
import shutil
from pathlib import Path

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")  # one GPU only (Kaggle T4 x2 would otherwise split the model)

import torch

ON_KAGGLE = Path("/kaggle/working").exists()
SMOKE = os.environ.get("CIVICLOOP_SMOKE") == "1"  # tiny CPU run: checks the code path only
MODEL_ID = "trl-internal-testing/tiny-Qwen2ForCausalLM-2.5" if SMOKE else "Qwen/Qwen2.5-1.5B-Instruct"


def find_data_dir(input_root="/kaggle/input"):
    """CIVICLOOP_DATA if set, else the folder under the Kaggle input root that holds sft_train.jsonl."""
    env = os.environ.get("CIVICLOOP_DATA")
    if env:
        return Path(env)
    root = Path(input_root)
    if root.exists():
        hits = sorted(root.rglob("sft_train.jsonl"))
        if hits:
            return hits[0].parent
    return Path(".")


DATA = find_data_dir()
OUT = Path(os.environ.get("CIVICLOOP_OUT", "triage_llm"))
SEED = 42  # = config.SEED
random.seed(SEED)
torch.manual_seed(SEED)
GPU = torch.cuda.is_available()
assert GPU or SMOKE, "No GPU: Colab Runtime -> Change runtime type -> T4 GPU; Kaggle Settings -> Accelerator"
OUT.mkdir(parents=True, exist_ok=True)
print("data folder:", DATA.resolve(), "| gpu:", torch.cuda.get_device_name(0) if GPU else "none")

In [ ]:
FILES = ["sft_train.jsonl", "dpo_train.jsonl", "eval_prompts.jsonl"]
if not all((DATA / f).exists() for f in FILES):
    if ON_KAGGLE:
        tree = sorted(str(p) for p in Path("/kaggle/input").rglob("*"))[:40]
        raise FileNotFoundError(
            f"Looked in {DATA.resolve()} and did not find all of {FILES}.\n"
            "Add the dataset with Add Input (right panel). What is mounted under /kaggle/input:\n"
            + "\n".join(tree))
    from google.colab import files  # Colab only
    files.upload()
missing = [f for f in FILES if not (DATA / f).exists()]
assert not missing, f"missing: {missing}"
print("found all 3 files in", DATA.resolve())

In [ ]:
from datasets import load_dataset


def load(name, n=None):
    ds = load_dataset("json", data_files=str(DATA / name), split="train")
    return ds.select(range(min(n, len(ds)))) if n else ds


sft_ds = load("sft_train.jsonl", 32 if SMOKE else None)
dpo_ds = load("dpo_train.jsonl", 8 if SMOKE else None).remove_columns(["case_id", "kind", "official_id"])
with open(DATA / "eval_prompts.jsonl") as f:
    eval_rows = [json.loads(line) for line in f][: 8 if SMOKE else None]
print(f"SFT {len(sft_ds)} rows, DPO {len(dpo_ds)} pairs, eval {len(eval_rows)} prompts")
print(sft_ds[0]["prompt"][1]["content"], "\n->", sft_ds[0]["completion"][0]["content"])

In [ ]:
from peft import LoraConfig, PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL_ID)


def base_model():
    """The frozen base. fp16 on the GPU (a T4 has no bf16), fp32 for the CPU smoke run."""
    dtype = torch.float16 if GPU else torch.float32
    return AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=dtype,
                                                device_map={"": 0} if GPU else None)


lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules="all-linear",
                  task_type="CAUSAL_LM")
common = dict(seed=SEED, report_to="none", save_strategy="no", logging_steps=1 if SMOKE else 10,
              fp16=GPU, use_cpu=not GPU, gradient_checkpointing=GPU,
              max_steps=2 if SMOKE else -1)


def generate(model, rows, batch=32):
    tok.padding_side = "left"
    model.eval()
    outs = []
    for i in range(0, len(rows), batch):
        texts = [tok.apply_chat_template(r["prompt"], add_generation_prompt=True, tokenize=False)
                 for r in rows[i:i + batch]]
        enc = tok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        with torch.no_grad():
            gen = model.generate(**enc, max_new_tokens=128, do_sample=False,
                                 pad_token_id=tok.pad_token_id)
        outs += tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return outs


def severity_of(text):
    """The severity word of one answer, or "invalid" (full parsing is local: llm_data.parse)."""
    try:
        return json.loads(text.strip().removeprefix("```json").removesuffix("```")).get("severity", "invalid")
    except (json.JSONDecodeError, AttributeError):
        return "invalid"


In [ ]:
# Stage 1: SFT. Loss only on the answer tokens (prompt-completion data), 1 epoch.
from trl import SFTConfig, SFTTrainer

sft_args = SFTConfig(output_dir=str(OUT / "sft_run"), num_train_epochs=1,
                     per_device_train_batch_size=16, learning_rate=2e-4,
                     lr_scheduler_type="cosine", warmup_steps=10, max_length=512, **common)
sft = SFTTrainer(model=base_model(), args=sft_args, train_dataset=sft_ds, processing_class=tok,
                 peft_config=lora)
sft.train()
sft.model.save_pretrained(OUT / "sft")
logs = {"sft": sft.state.log_history}
del sft
torch.cuda.empty_cache() if GPU else None

In [ ]:
# Stage 2: DPO. Start from the SFT adapter (trainable); TRL copies it as a frozen "ref"
# adapter, so the reference is the SFT model, not the raw base.
# Run 3 (runs 1 and 2 collapsed to "high", then "critical"):
#   loss = DPO + SFT on the chosen answer (1:1). DPO alone only widens the chosen-rejected gap
#   and can do it by lowering both, which frees probability for an answer no pair rejects.
#   The SFT term keeps the chosen answers likely. Also gentler: lr 1e-5, 1 epoch.
# Watch rewards/chosen (should stay near or above 0), rewards/accuracies and rewards/margins.
from collections import Counter

from trl import DPOConfig, DPOTrainer

policy = PeftModel.from_pretrained(base_model(), OUT / "sft", is_trainable=True)
dpo_args = DPOConfig(output_dir=str(OUT / "dpo_run"), num_train_epochs=1,
                     per_device_train_batch_size=8, learning_rate=1e-5, beta=0.1,
                     loss_type=["sigmoid", "sft"], loss_weights=[1.0, 1.0],
                     max_length=512, **common)
dpo = DPOTrainer(model=policy, args=dpo_args, train_dataset=dpo_ds, processing_class=tok)
dpo.train()
dpo.model.save_pretrained(OUT / "dpo", selected_adapters=["default"])
logs["dpo"] = dpo.state.log_history
with open(OUT / "train_log.json", "w") as f:
    json.dump(logs, f, indent=1)

# Collapse check before spending time on 3000 answers: 150 random eval prompts, stop if one
# severity takes more than 60% of them.
check_rows = random.Random(SEED).sample(eval_rows, min(150, len(eval_rows)))
dpo.model.set_adapter("default")  # the trained policy, not the frozen "ref" copy
counts = Counter(severity_of(o) for o in generate(dpo.model, check_rows))
print(f"DPO severities on {len(check_rows)} eval prompts:", dict(counts.most_common()))
top, n = counts.most_common(1)[0]
if n > 0.6 * len(check_rows) and not SMOKE:  # the tiny smoke model writes nonsense anyway
    raise RuntimeError(f"DPO collapsed: {top!r} is {n / len(check_rows):.0%} of answers. "
                       "Run 3 was the last: record the negative result and move to 3.6.")
del dpo, policy
torch.cuda.empty_cache() if GPU else None


In [ ]:
# Stage 3: greedy answers for every eval prompt, from both adapters on one base.
model = PeftModel.from_pretrained(base_model(), OUT / "sft", adapter_name="llm-sft")
model.load_adapter(OUT / "dpo", adapter_name="llm-dpo")
preds = []
for version in ("llm-sft", "llm-dpo"):
    model.set_adapter(version)
    for r, out in zip(eval_rows, generate(model, eval_rows)):
        preds.append({"case_id": r["case_id"], "set": r["set"], "version": version, "output": out})
with open(OUT / "predictions.jsonl", "w") as f:
    for p in preds:
        f.write(json.dumps(p, ensure_ascii=False) + "\n")
print(f"{len(preds)} answers")

In [ ]:
# Sanity check: different cases must get different explanations (an input-blind model
# writes the same fluent answer every time). Full scoring is local: trainer.llm_eval.
for p in preds[:3] + [p for p in preds if p["set"] == "holdout"][:2]:
    print(p["version"], p["case_id"], "->", p["output"])

In [ ]:
for run in ("sft_run", "dpo_run"):
    shutil.rmtree(OUT / run, ignore_errors=True)
zip_path = shutil.make_archive("triage_llm", "zip", OUT)
print("wrote", Path(zip_path).resolve())
if ON_KAGGLE:
    print("Kaggle: open the Output panel -> /kaggle/working -> triage_llm.zip -> download")
else:
    try:
        from google.colab import files
        files.download("triage_llm.zip")
    except ImportError:
        print("Download triage_llm.zip from the file browser")